# Apple Leaf Disease Classification: Academic Machine Learning Workflow
---
This notebook demonstrates the end-to-end academic machine learning pipeline for **4-class apple leaf disease classification** using the PlantVillage dataset.

### Methodology Pipeline:
1. **Dataset Loading**: PlantVillage Apple subset (4 classes: Apple Scab, Black Rot, Cedar Apple Rust, Healthy)
2. **Image Preprocessing**: Resizing (500x500), BGR to RGB/HSV conversion, and dual-color mask HSV segmentation
3. **Feature Extraction**: Combined handcrafted descriptors:
   - Hu Moments (7 shape invariants)
   - Haralick Texture Features (13 GLCM statistical descriptors)
   - HSV 3D Color Histogram (512 bins)
   - Total Feature Vector: **532 dimensions**
4. **Model Training & Comparison**: Random Forest vs. Support Vector Machine (SVM with MinMaxScaler fitted solely on training data)
5. **Evaluation & Diagnosis**: Accuracy, Precision, Recall, F1-Score, Confusion Matrices, and live prediction.


In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
import mahotas

# Bootstrap project root
sys.path.append(os.path.abspath('..'))
from src.preprocessing import preprocess_image, load_image, segment_leaf
from src.feature_extraction import extract_image_features, CLASS_MAP
from src.predict import AppleLeafPredictor

print('Environment and modules loaded successfully!')

## 1. Dataset Inspection & Class Distribution
The dataset consists of 3,171 apple leaf photographs categorized into four classes.

In [ ]:
data_dir = os.path.join('..', 'data', 'raw')
class_dirs = sorted([d for d in os.listdir(data_dir) if os.path.isdir(os.path.join(data_dir, d))])

counts = {}
for d in class_dirs:
    readable = CLASS_MAP.get(d, d)
    count = len(os.listdir(os.path.join(data_dir, d)))
    counts[readable] = count

df_dist = pd.DataFrame(list(counts.items()), columns=['Class', 'Sample Count'])
print(df_dist.to_string(index=False))

plt.figure(figsize=(8, 4), dpi=150)
sns.barplot(data=df_dist, x='Class', y='Sample Count', palette='viridis')
plt.title('Class Distribution in PlantVillage Apple Dataset')
plt.ylabel('Number of Images')
plt.show()

## 2. Image Preprocessing & HSV Color Segmentation
We isolate the leaf tissue (healthy green and necrotic/rust brown) from the background using HSV thresholding.

In [ ]:
sample_cls = 'Apple___Cedar_apple_rust'
sample_file = os.listdir(os.path.join(data_dir, sample_cls))[0]
sample_path = os.path.join(data_dir, sample_cls, sample_file)

prep = preprocess_image(sample_path)

fig, axes = plt.subplots(1, 4, figsize=(16, 4), dpi=150)
axes[0].imshow(prep['original_rgb'])
axes[0].set_title('Original Leaf (500x500)')
axes[0].axis('off')

axes[1].imshow(prep['hsv'])
axes[1].set_title('HSV Representation')
axes[1].axis('off')

axes[2].imshow(prep['mask'], cmap='gray')
axes[2].set_title('Combined Dual Mask')
axes[2].axis('off')

axes[3].imshow(prep['segmented_rgb'])
axes[3].set_title('Segmented Leaf')
axes[3].axis('off')

plt.tight_layout()
plt.show()

## 3. Combined Feature Extraction (532 Dimensions)
We extract Hu Moments (7), Haralick Textures (13), and HSV 3D Color Histograms (512).

In [ ]:
fv = extract_image_features(sample_path)
print(f'Dynamically extracted feature vector dimension: {fv.shape[0]}')
print(' - HSV Histogram bins (8x8x8): 512 features')
print(' - Haralick Texture Descriptors: 13 features')
print(' - Hu Moment Invariants:        7 features')
print(f'Total: 512 + 13 + 7 = {fv.shape[0]} features')

## 4. Model Evaluation & Comparison (Random Forest vs. SVM)
We load the test set evaluation results from `results/model_comparison.csv`.

In [ ]:
comp_csv = os.path.join('..', 'results', 'model_comparison.csv')
comp_df = pd.read_csv(comp_csv)
print('\n--- MODEL COMPARISON TABLE ---')
print(comp_df[['model', 'accuracy', 'precision', 'recall', 'f1_score']].to_string(index=False))

# Plot comparison
plt.figure(figsize=(8, 5), dpi=150)
plot_df = comp_df.melt(id_vars=['model'], value_vars=['accuracy', 'precision', 'recall', 'f1_score'], var_name='Metric', value_name='Score')
sns.barplot(data=plot_df, x='Metric', y='Score', hue='model', palette='tab10')
plt.ylim(0.85, 1.0)
plt.title('Random Forest vs SVM Performance on Held-Out Test Set (N=635)')
plt.show()

## 5. Live Inference Demonstration
Testing classification on an unseen leaf sample using `AppleLeafPredictor`.

In [ ]:
predictor = AppleLeafPredictor(models_dir=os.path.join('..', 'models'))
res_rf = predictor.predict(sample_path, model_name='random_forest')
res_svm = predictor.predict(sample_path, model_name='svm')

print(f'Sample: {sample_file}')
print(f"Random Forest Prediction: {res_rf['predicted_class']} ({res_rf['confidence_percent']:.2f}% confidence)")
print(f"SVM Prediction:           {res_svm['predicted_class']} ({res_svm['confidence_percent']:.2f}% confidence)")